In [1]:
import os
import numpy as np
import pandas as pd
from IPython.display import display, HTML
from sklearn.utils import shuffle
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

In [2]:
# contains dicom ID - 377110
df_metadata = pd.read_csv("/vol/biodata/data/chest_xray/mimic-cxr-jpg/mimic-cxr-2.0.0-metadata.csv.gz")
# we are going to ignore this as split is train:97.5 test:1.5 and valid:0.75 which is really small for our purpose
# df_split = pd.read_csv("/vol/biodata/data/chest_xray/mimic-cxr-jpg/mimic-cxr-2.0.0-split.csv.gz")

# based on CheXMask - only dicom ID - 243334
df_seg = pd.read_csv("/data2/rmehta3/datasets/chexmask-cxr-segmentation-data/0.4/Preprocessed/MIMIC-CXR-JPG.csv")

# contains subject ID only - 227827
# df_negbio = pd.read_csv("/vol/biodata/data/chest_xray/mimic-cxr-jpg/mimic-cxr-2.0.0-negbio.csv.gz")
df_chexpert = pd.read_csv("/vol/biodata/data/chest_xray/mimic-cxr-jpg/mimic-cxr-2.0.0-chexpert.csv.gz")

# metdata - admission - 523740 - subject_ID
df_admission = pd.read_csv("/vol/biodata/data/chest_xray/mimic-iv/mimic-iv-1.0/core/admissions.csv")

# metadata - patient - 382278 - subject_ID 
df_patient = pd.read_csv("/vol/biodata/data/chest_xray/mimic-iv/mimic-iv-1.0/core/patients.csv.gz")

In [ ]:
############
print(f"MetaData DF len: {len(df_metadata)}")
print(f"MetaData DF Columns: {df_metadata.columns}")


##########
print(f"Seg DF len: {len(df_seg)}")
print(f"Seg DF Columns: {df_seg.columns}")


print(f"ChexPert DF len: {len(df_chexpert)}")
print(f"ChexPert DF Columns: {df_chexpert.columns}")

#########
print(f"Admission DF len: {len(df_admission)}")
print(f"Admission DF Columns: {df_admission.columns}")

print(f"Patient DF len: {len(df_patient)}")
print(f"Patient DF Columns: {df_patient.columns}")


MetaData DF len: 377110
MetaData DF Columns: Index(['dicom_id', 'subject_id', 'study_id',
       'PerformedProcedureStepDescription', 'ViewPosition', 'Rows', 'Columns',
       'StudyDate', 'StudyTime', 'ProcedureCodeSequence_CodeMeaning',
       'ViewCodeSequence_CodeMeaning',
       'PatientOrientationCodeSequence_CodeMeaning'],
      dtype='object')
Seg DF len: 243334
Seg DF Columns: Index(['dicom_id', 'Dice RCA (Mean)', 'Dice RCA (Max)', 'Landmarks',
       'Left Lung', 'Right Lung', 'Heart', 'Height', 'Width'],
      dtype='object')
ChexPert DF len: 227827
ChexPert DF Columns: Index(['subject_id', 'study_id', 'Atelectasis', 'Cardiomegaly',
       'Consolidation', 'Edema', 'Enlarged Cardiomediastinum', 'Fracture',
       'Lung Lesion', 'Lung Opacity', 'No Finding', 'Pleural Effusion',
       'Pleural Other', 'Pneumonia', 'Pneumothorax', 'Support Devices'],
      dtype='object')
Admission DF len: 523740
Admission DF Columns: Index(['subject_id', 'hadm_id', 'admittime', 'dischtime', '

In [4]:
print("Number of images: " + str(len(df_metadata)))
print("Number of patients: " + str(df_metadata.subject_id.nunique()))

Number of images: 377110
Number of patients: 65379


In [ ]:
df_ethnicity = df_admission.loc[:,['subject_id', 'ethnicity']].drop_duplicates()

v = df_ethnicity.subject_id.value_counts()
subject_id_more_than_once = v.index[v.gt(1)]

df_ambiguous_ethnicity = df_ethnicity[df_ethnicity.subject_id.isin(subject_id_more_than_once)]
inconsistent_race = df_ambiguous_ethnicity.subject_id.unique()

grouped = df_ambiguous_ethnicity.groupby('subject_id')

In [6]:
df_merge = pd.merge(df_metadata,df_chexpert,on=['subject_id', 'study_id'])
df_merge = pd.merge(df_merge,df_ethnicity,on='subject_id')
df_merge = df_merge[~df_merge.subject_id.isin(inconsistent_race)]
df_merge = df_merge.rename(columns={"ethnicity": "race"})
df_merge = df_merge[df_merge.race.isin(['ASIAN','BLACK/AFRICAN AMERICAN','WHITE'])]

In [7]:
df_cxr = pd.merge(df_merge,df_patient, on='subject_id')

In [8]:
df_cxr = df_cxr.rename(columns={'gender': 'sex'})
df_cxr = df_cxr.rename(columns={'anchor_age': 'age'})

study_year = np.floor(df_cxr['StudyDate'] / 10000)
delta_years = study_year - df_cxr['anchor_year']
df_cxr['age'] = df_cxr['age'] + delta_years

In [9]:
white = 'White'
asian = 'Asian'
black = 'Black'

mask = (df_cxr.race.str.contains("BLACK", na=False))
df_cxr.loc[mask, "race"] = black

mask = (df_cxr.race.str.contains("WHITE", na=False))
df_cxr.loc[mask, "race"] = white

mask = (df_cxr.race.str.contains("ASIAN", na=False))
df_cxr.loc[mask, "race"] = asian

df_cxr = df_cxr[df_cxr.race.isin([asian,black,white])]

df_cxr['race_label'] = df_cxr['race']

df_cxr.loc[df_cxr['race_label'] == white, 'race_label'] = 0 # White 
df_cxr.loc[df_cxr['race_label'] == asian, 'race_label'] = 1 # Asian
df_cxr.loc[df_cxr['race_label'] == black, 'race_label'] = 2 # Black

In [10]:
df_cxr.loc[df_cxr['sex'] == 'F', 'sex'] = 'Female'
df_cxr.loc[df_cxr['sex'] == 'M', 'sex'] = 'Male'

df_cxr['sex_label'] = df_cxr['sex']

df_cxr.loc[df_cxr['sex_label'] == 'Male', 'sex_label'] = 0    # Male
df_cxr.loc[df_cxr['sex_label'] == 'Female', 'sex_label'] = 1  # Female


In [ ]:
df_cxr = df_cxr[df_cxr.ViewPosition.isin(['AP','PA'])]

df_cxr['ViewPosition_label'] = df_cxr['ViewPosition']

df_cxr.loc[df_cxr['ViewPosition_label'] == 'AP', 'ViewPosition_label'] = 0 # AP 
df_cxr.loc[df_cxr['ViewPosition_label'] == 'PA', 'ViewPosition_label'] = 1 # PA

In [ ]:
labels = [
    'No Finding',
    'Enlarged Cardiomediastinum', #enlargement of the cardiac silhouette
    'Cardiomegaly', # enlargment of the heart
    'Lung Opacity',
    'Lung Lesion',
    'Edema',
    'Consolidation',
    'Pneumonia',
    'Atelectasis',
    'Pneumothorax',
    'Pleural Effusion',
    'Pleural Other',
    'Fracture',
    'Support Devices']


print(f"No Finding unique values: {df_cxr['No Finding'].unique()}")
print(f"Enlarged Cardiomediastinum unique values: {df_cxr['Enlarged Cardiomediastinum'].unique()}")
print(f"Cardiomegaly unique values: {df_cxr['Cardiomegaly'].unique()}")
print(f"Lung Opacity unique values: {df_cxr['Lung Opacity'].unique()}")
print(f"Lung Lesion unique values: {df_cxr['Lung Lesion'].unique()}")
print(f"Edema unique values: {df_cxr['Edema'].unique()}")
print(f"Consolidation unique values: {df_cxr['Consolidation'].unique()}")
print(f"Pneumonia unique values: {df_cxr['Pneumonia'].unique()}")
print(f"Atelectasis unique values: {df_cxr['Atelectasis'].unique()}")
print(f"Pneumothorax unique values: {df_cxr['Pneumothorax'].unique()}")
print(f"Pleural Effusion unique values: {df_cxr['Pleural Effusion'].unique()}")
print(f"Pleural Other unique values: {df_cxr['Pleural Other'].unique()}")
print(f"Fracture unique values: {df_cxr['Fracture'].unique()}")
print(f"Support Devices unique values: {df_cxr['Support Devices'].unique()}")


print("\n\n")

print(f"No Finding patients: {len(df_cxr[df_cxr['No Finding']==1])}")
print(f"Enlarged Cardiomediastinum patients: {len(df_cxr[df_cxr['Enlarged Cardiomediastinum']==1])}")
print(f"Cardiomegaly patients: {len(df_cxr[df_cxr['Cardiomegaly']==1])}")
print(f"Lung Opacity patients: {len(df_cxr[df_cxr['Lung Opacity']==1])}")
print(f"Lung Lesion patients: {len(df_cxr[df_cxr['Lung Lesion']==1])}")
print(f"Edema patients: {len(df_cxr[df_cxr['Edema']==1])}")
print(f"Consolidation patients: {len(df_cxr[df_cxr['Consolidation']==1])}")
print(f"Pneumonia patients: {len(df_cxr[df_cxr['Pneumonia']==1])}")
print(f"Atelectasis patients: {len(df_cxr[df_cxr['Atelectasis']==1])}")
print(f"Pneumothorax patients: {len(df_cxr[df_cxr['Pneumothorax']==1])}")
print(f"Pleural Effusion patients: {len(df_cxr[df_cxr['Pleural Effusion']==1])}")
print(f"Pleural Other patients: {len(df_cxr[df_cxr['Pleural Other']==1])}")
print(f"Fracture patients: {len(df_cxr[df_cxr['Fracture']==1])}")
print(f"Support Devices patients: {len(df_cxr[df_cxr['Support Devices']==1])}")

# used in ICML paper of Fabio/Tian and Ben's paper on fairness
# ignored last 
print("\n\n")
print("NoFinding=1 :",len(df_cxr[(df_cxr['No Finding']==1)]))
print("NoFinding!=1 and PE=1 :",len(df_cxr[(df_cxr['No Finding']!=1) & (df_cxr['Pleural Effusion']==1)]))
print("NoFinding!=1 and PE!=1 :",len(df_cxr[(df_cxr['No Finding']!=1) & (df_cxr['Pleural Effusion']!=1)]))
## only PE
print("\n\n")
print("NoFinding=1 :",len(df_cxr[(df_cxr['No Finding']==1)]))
print("NoFinding!=1 and PE=1 and OD!=1 :",len(df_cxr[(df_cxr['No Finding']!=1) & (df_cxr['Pleural Effusion']==1) & (df_cxr['Cardiomegaly']!=1) & (df_cxr['Enlarged Cardiomediastinum']!=1) & (df_cxr['Lung Opacity']!=1) & (df_cxr['Lung Lesion']!=1) & (df_cxr['Edema']!=1) & (df_cxr['Consolidation']!=1) & (df_cxr['Pneumonia']!=1) & (df_cxr['Atelectasis']!=1) & (df_cxr['Pneumothorax']!=1) & (df_cxr['Pleural Other']!=1) ])) # 1
## only LO - no enlarged cardiomediastinum, cardiomegaly, lung lesion, atelectasis, pneumothorax, pleural others, fracture. includes pleural effusion, consolidation, edema, pneumonia

No Finding unique values: [ 1. nan]
Enlarged Cardiomediastinum unique values: [nan  0. -1.  1.]
Cardiomegaly unique values: [nan  0.  1. -1.]
Lung Opacity unique values: [nan -1.  1.  0.]
Lung Lesion unique values: [nan  1.  0. -1.]
Edema unique values: [nan -1.  0.  1.]
Consolidation unique values: [nan  1. -1.  0.]
Pneumonia unique values: [nan -1.  1.  0.]
Atelectasis unique values: [nan -1.  1.  0.]
Pneumothorax unique values: [nan  1.  0. -1.]
Pleural Effusion unique values: [nan  1.  0. -1.]
Pleural Other unique values: [nan  1. -1.  0.]
Fracture unique values: [nan  1.  0. -1.]
Support Devices unique values: [nan  1.  0. -1.]



No Finding patients: 56615
Enlarged Cardiomediastinum patients: 5923
Cardiomegaly patients: 37301
Lung Opacity patients: 42738
Lung Lesion patients: 5396
Edema patients: 23043
Consolidation patients: 8948
Pneumonia patients: 13377
Atelectasis patients: 38146
Pneumothorax patients: 8906
Pleural Effusion patients: 46224
Pleural Other patients: 1696
Fractur

In [ ]:
# just saving it for visualization of all images
df_cxr_seg = pd.merge(df_cxr,df_seg,on=['dicom_id'],how='inner')
df_cxr_seg.to_csv("/data2/rmehta3/datasets/chest_xray/all.csv", index=False)


In [ ]:

# # df_cxr['disease'] = df_cxr['No Finding']
# # df_cxr.loc[df_cxr['No Finding']==1, 'disease'] = 'No Finding'
# # df_cxr.loc[(df_cxr['No Finding']!=1) & (df_cxr['Pleural Effusion']==1), 'disease'] = 'Pleural Effusion'


# # 56614 - 6005
# only NoFinding and PE (no other disease)
df_cxr['disease'] = df_cxr['No Finding']
df_cxr.loc[df_cxr['No Finding']==1, 'disease'] = 'No Finding'
df_cxr.loc[(df_cxr['No Finding']!=1) & (df_cxr['Pleural Effusion']==1) & (df_cxr['Cardiomegaly']!=1) & (df_cxr['Enlarged Cardiomediastinum']!=1) & (df_cxr['Lung Opacity']!=1) & (df_cxr['Lung Lesion']!=1) & (df_cxr['Edema']!=1) & (df_cxr['Consolidation']!=1) & (df_cxr['Pneumonia']!=1) & (df_cxr['Atelectasis']!=1) & (df_cxr['Pneumothorax']!=1) & (df_cxr['Pleural Other']!=1), 'disease'] = 'Pleural Effusion'


df_cxr['disease_label'] = df_cxr['disease']
df_cxr.loc[df_cxr['disease_label'] == 'No Finding', 'disease_label'] = 0
df_cxr.loc[df_cxr['disease_label'] == 'Pleural Effusion', 'disease_label'] = 1


df_cxr = df_cxr[df_cxr.disease_label.isin([0,1])]
df_cxr.disease_label.value_counts()

/tmp/ipykernel_3332321/956988273.py:53: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'No Finding' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_cxr.loc[df_cxr['No Finding']==1, 'disease'] = 'No Finding'


disease_label
0    56615
1    27423
Name: count, dtype: int64

In [ ]:
df_cxr_seg = pd.merge(df_cxr,df_seg,on=['dicom_id'],how='inner')

In [15]:
df_cxr_seg.columns

Index(['dicom_id', 'subject_id', 'study_id',
       'PerformedProcedureStepDescription', 'ViewPosition', 'Rows', 'Columns',
       'StudyDate', 'StudyTime', 'ProcedureCodeSequence_CodeMeaning',
       'ViewCodeSequence_CodeMeaning',
       'PatientOrientationCodeSequence_CodeMeaning', 'Atelectasis',
       'Cardiomegaly', 'Consolidation', 'Edema', 'Enlarged Cardiomediastinum',
       'Fracture', 'Lung Lesion', 'Lung Opacity', 'No Finding',
       'Pleural Effusion', 'Pleural Other', 'Pneumonia', 'Pneumothorax',
       'Support Devices', 'race', 'sex', 'age', 'anchor_year',
       'anchor_year_group', 'dod', 'race_label', 'sex_label',
       'ViewPosition_label', 'disease', 'disease_label', 'Dice RCA (Mean)',
       'Dice RCA (Max)', 'Landmarks', 'Left Lung', 'Right Lung', 'Heart',
       'Height', 'Width'],
      dtype='object')

In [ ]:
# we only want to keep following columns

df_cxr_seg = df_cxr_seg.loc[:,['dicom_id',
                                'subject_id',
                                'study_id',
                                'age',
                                'ViewPosition',        # AP, PA
                                'ViewPosition_label',  # 0, 1
                                'race',                # White, Asian, Black
                                'race_label',          # 0, 1, 2
                                'sex',                 # Male, Female
                                'sex_label',           # 0, 1
                                'disease',             # NoFinding, PF
                                'disease_label',       # 0, 1
                                'Dice RCA (Mean)',
                                'Dice RCA (Max)']]

In [17]:
df_cxr_seg.insert(4, "split","none", True)

# split based on subject ID
unique_sub_id = df_cxr_seg.subject_id.unique()

train_percent, valid_percent, test_percent = 0.60, 0.10, 0.30

unique_sub_id = shuffle(unique_sub_id)
value1 = (round(len(unique_sub_id)*train_percent))
value2 = (round(len(unique_sub_id)*valid_percent))
value3 = value1 + value2
value4 = (round(len(unique_sub_id)*test_percent))

print(f"Total Patients: {len(unique_sub_id)}")
print(f"Patients in training set: {value1}")
print(f"Patients in validation set: {value2}")
print(f"Patients in testing set: {value4} ")

# assign split for each dicom_id based on its corresponding subject_id
df_cxr_seg = shuffle(df_cxr_seg)
train_sub_id = unique_sub_id[:value1]
validate_sub_id = unique_sub_id[value1:value3]
test_sub_id = unique_sub_id[value3:]

df_cxr_seg.loc[df_cxr_seg.subject_id.isin(train_sub_id), "split"]="train"
df_cxr_seg.loc[df_cxr_seg.subject_id.isin(validate_sub_id), "split"]="validate"
df_cxr_seg.loc[df_cxr_seg.subject_id.isin(test_sub_id), "split"]="test"

df_cxr_seg.split.value_counts(normalize=True)
df_cxr_seg.split.value_counts(normalize=False)

Total Patients: 32903
Patients in training set: 19742
Patients in validation set: 3290
Patients in testing set: 9871 


split
train       50272
test        25564
validate     8202
Name: count, dtype: int64

In [18]:
df_train = df_cxr_seg[df_cxr_seg.split == "train"]
df_test = df_cxr_seg[df_cxr_seg.split == "test"]
df_valid = df_cxr_seg[df_cxr_seg.split == "validate"]

In [21]:
df_train.disease_label.value_counts()

disease_label
0    33936
1    16336
Name: count, dtype: int64

In [ ]:

# df_test.to_csv("/data2/rmehta3/datasets/chest_xray/test_pe.csv", index=False)
# df_train.to_csv("/data2/rmehta3/datasets/chest_xray/train_pe.csv", index=False)
# df_valid.to_csv("/data2/rmehta3/datasets/chest_xray/valid_pe.csv", index=False)

df_test.to_csv("/data2/rmehta3/datasets/chest_xray/test_pe_only.csv", index=False)
df_train.to_csv("/data2/rmehta3/datasets/chest_xray/train_pe_only.csv", index=False)
df_valid.to_csv("/data2/rmehta3/datasets/chest_xray/valid_pe_only.csv", index=False)
